# 🧠 Taller SQL - Gimnasia Rítmica con PostgreSQL

Este taller está diseñado para ejecutarse en Google Colaboratory usando una base de datos PostgreSQL. Aprenderás a crear tablas, insertar datos y realizar consultas SQL avanzadas sobre una base de datos que modela competencias de gimnasia rítmica.

## Objetivos
- Crear una base de datos PostgreSQL en Colab.
- Crear tablas con claves primarias y foráneas.
- Insertar datos simulados.
- Ejecutar consultas SQL avanzadas.
- Resolver ejercicios prácticos.


In [ ]:
# Instalación y configuración de PostgreSQL
!apt update
!apt install postgresql postgresql-contrib -y
!service postgresql start
!sudo -u postgres psql -c "CREATE USER root WITH SUPERUSER"


In [ ]:
# Cargar extensión SQL y conectar con PostgreSQL
%load_ext sql
%config SqlMagic.feedback=False
%config SqlMagic.autopandas=True
%sql postgresql+psycopg2://@/postgres


## 🏗️ Crear tablas

Creamos las tablas necesarias para modelar la base de datos de gimnasia rítmica. Incluye campos tipo `DATE` y relaciones entre entidades.


In [ ]:
%%sql

DROP TABLE IF EXISTS Evaluaciones, Presentaciones, Gimnastas, Conjuntos, Instrumentos, Campeonatos, Tipos, Equipos CASCADE;

CREATE TABLE Equipos (
    id_equipo SERIAL PRIMARY KEY,
    nombre_equipo TEXT
);

CREATE TABLE Conjuntos (
    id_conj SERIAL PRIMARY KEY,
    nombre_conj TEXT,
    id_equipo INTEGER REFERENCES Equipos(id_equipo)
);

CREATE TABLE Gimnastas (
    id_gim SERIAL PRIMARY KEY,
    nombre TEXT,
    fecha_nac DATE,
    id_conj INTEGER REFERENCES Conjuntos(id_conj),
    id_equipo INTEGER REFERENCES Equipos(id_equipo)
);

CREATE TABLE Instrumentos (
    id_inst SERIAL PRIMARY KEY,
    nombre_inst TEXT
);

CREATE TABLE Campeonatos (
    id_camp SERIAL PRIMARY KEY,
    nombre_camp TEXT,
    fecha_camp DATE
);

CREATE TABLE Tipos (
    id_tipo SERIAL PRIMARY KEY,
    nombre_tipo TEXT
);

CREATE TABLE Presentaciones (
    id_pres SERIAL PRIMARY KEY,
    id_gim INTEGER REFERENCES Gimnastas(id_gim),
    id_conj INTEGER REFERENCES Conjuntos(id_conj),
    id_inst INTEGER REFERENCES Instrumentos(id_inst),
    id_camp INTEGER REFERENCES Campeonatos(id_camp),
    id_tipo INTEGER REFERENCES Tipos(id_tipo)
);

CREATE TABLE Evaluaciones (
    id_eval SERIAL PRIMARY KEY,
    id_pres INTEGER REFERENCES Presentaciones(id_pres),
    pje_eje REAL,
    pje_dif REAL,
    pje_art REAL
);


In [ ]:
%config SqlMagic.style = '_DEPRECATED_DEFAULT'


## 📥 Insertar datos simulados

Insertamos datos ficticios para poblar la base de datos: 5 equipos, 10 conjuntos, 20 gimnastas, 15 campeonatos, 50 presentaciones individuales, 50 presentaciones de conjunto y 100 evaluaciones.


In [ ]:
%%sql

-- Insertar equipos
INSERT INTO Equipos (nombre_equipo) VALUES
('Estrellas'), ('Auroras'), ('Fénix'), ('Orion'), ('Galaxia');

-- Insertar conjuntos
INSERT INTO Conjuntos (nombre_conj, id_equipo) VALUES
('Estrellas A', 1), ('Estrellas B', 1), ('Auroras A', 2), ('Auroras B', 2),
('Fénix A', 3), ('Fénix B', 3), ('Orion A', 4), ('Orion B', 4),
('Galaxia A', 5), ('Galaxia B', 5);

-- Insertar gimnastas
INSERT INTO Gimnastas (nombre, fecha_nac, id_conj, id_equipo) VALUES
('Ana Torres', '2005-03-12', 1, 1), ('Lucía Pérez', '2006-07-25', 1, 1),
('María Gómez', '2004-11-05', 2, 1), ('Sofía Díaz', '2007-01-20', 2, 1),
('Valentina Ruiz', '2005-06-30', 3, 2), ('Camila Soto', '2006-09-15', 3, 2),
('Isabella León', '2004-12-01', 4, 2), ('Martina Ríos', '2007-04-10', 4, 2),
('Renata Silva', '2005-08-22', 5, 3), ('Emilia Vargas', '2006-10-05', 5, 3),
('Josefa Herrera', '2004-07-18', 6, 3), ('Antonia Castro', '2007-02-28', 6, 3),
('Florencia Peña', '2005-05-14', 7, 4), ('Amanda Fuentes', '2006-11-11', 7, 4),
('Julieta Navarro', '2004-09-09', 8, 4), ('Agustina Bravo', '2007-03-03', 8, 4),
('Daniela Pino', '2005-12-25', 9, 5), ('Bianca Morales', '2006-06-06', 9, 5),
('Carla Espinoza', '2004-10-10', 10, 5), ('Fernanda Reyes', '2007-07-07', 10, 5),
('Colomba Gómez', '2004-11-05', 2, 1), ('Montserrat Díaz', '2007-01-20', 2, 1);

-- Insertar instrumentos
INSERT INTO Instrumentos (nombre_inst) VALUES
('Cinta'), ('Aro'), ('Balón'), ('Manos libres'), ('Cuerda');

-- Insertar campeonatos
INSERT INTO Campeonatos (nombre_camp, fecha_camp) VALUES
('Campeonato Nacional', '2023-06-15'), ('Copa Primavera', '2023-09-10'),
('Copa Invierno', '2023-12-05'), ('Torneo Sur', '2023-08-20'),
('Copa Norte', '2023-07-01'), ('Copa Andes', '2023-10-10'),
('Copa Pacífico', '2023-11-11'), ('Copa Estrellas', '2024-05-05'),
('Copa Aurora', '2024-04-04'), ('Copa Fénix', '2025-03-03'),
('Copa Orion', '2025-02-02'), ('Copa Galaxia', '2025-01-01'),
('Copa Final', '2024-12-31'), ('Copa Apertura', '2025-01-15'),
('Copa Clausura', '2024-12-01');

-- Insertar tipos
INSERT INTO Tipos (nombre_tipo) VALUES ('Individual'), ('Conjunto');




In [ ]:
%%sql
-- Insertar presentaciones
DO $$
DECLARE
    i INTEGER := 1;
    gim_id INTEGER;
    conj_id INTEGER;
    inst_id INTEGER;
    tipo_id INTEGER;
    camp_id INTEGER;
BEGIN
    WHILE i <= 100 LOOP
        -- Selección aleatoria de IDs válidos
        SELECT id_gim INTO gim_id FROM Gimnastas ORDER BY RANDOM() LIMIT 1;
        SELECT id_conj INTO conj_id FROM Conjuntos ORDER BY RANDOM() LIMIT 1;
        SELECT id_inst INTO inst_id FROM Instrumentos ORDER BY RANDOM() LIMIT 1;
        SELECT id_tipo INTO tipo_id FROM Tipos ORDER BY RANDOM() LIMIT 1;
        SELECT id_camp INTO camp_id FROM Campeonatos ORDER BY RANDOM() LIMIT 1;

        -- Inserción en la tabla Presentaciones
        INSERT INTO Presentaciones (id_gim, id_conj, id_inst, id_camp, id_tipo)
        VALUES (gim_id, conj_id, inst_id, camp_id, tipo_id);

        i := i + 1;
    END LOOP;
END $$;


In [ ]:
%%sql
-- Insertar evaluaciones
INSERT INTO Evaluaciones (id_pres, pje_eje, pje_dif, pje_art)
SELECT p.id_pres,
       ROUND((RANDOM() * 5 + 5)::NUMERIC, 2),
       ROUND((RANDOM() * 5 + 5)::NUMERIC, 2),
       ROUND((RANDOM() * 5 + 5)::NUMERIC, 2)
FROM Presentaciones p
LIMIT 100;

## 🔍 Ejercicios de Consultas SQL



1.- Gimnastas con más de 2 presentaciones individuales

In [ ]:
%%sql


2.- Conjuntos con más de 3 presentaciones

In [ ]:
%%sql


3.- Campeonatos en el segundo semestre del año 2023.

4.- Gimnastas ordenadas por fecha de nacimiento, desde la mayor a la menor.

5.- Promedio del puntaje total (suma del puntaje de ejecución, artístico y dificultad) por año y tipo de presentación.

6.- Máximo puntaje artístico por año y tipo de presentación

7.- Gimnastas que participaron en el campeonato más reciente

8.- Equipos que tienen más de 5 gimnastas

9.- Presentaciones individuales con el puntaje total más alto por año e instrumento.

10.- Presentaciones de conjunto con el puntaje total más alto por año e instrumento.

11.- Listar los nombres de los equipos y el promedio del puntaje artístico de todas las presentaciones (tanto individuales como por equipos) por año